# Normalizing flows in one dimension

Neural posterior estimation describes a posterior with a normalizing flow. This notebook introduces flows where
everything can be drawn and checked against the exact answer: one dimension. It shows what a flow is, the target
distribution of the course, the exact ("oracle") flow of that target, one fit with the package
`prob_reco_sbi.flows1d`, how to measure the fit, two animations (the training and the transport of samples) and the
uniform base. The maths is in [docs/flows.md](../docs/flows.md) and the tasks are in
[TASKS_flows.md](../TASKS_flows.md).

Run it in the environment of the [README](../README.md) with the kernel registered there; nothing new needs to be
installed. It takes about a minute on a CPU and writes two GIF files to the repository's `data/` folder, which is
`../data/` from this notebook's folder.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import LinearSegmentedColormap

import prob_reco_sbi
from prob_reco_sbi.flows1d import (
    CAMEL,
    Flow,
    RQSpline,
    Sigmoid,
    StandardNormal,
    Uniform,
    camel,
    fit,
    kl_to_truth,
    ks_statistic,
    oracle_map,
    qq_points,
)

DATA = Path("../data")  # the repository's data/ folder
DATA.mkdir(exist_ok=True)
print("prob_reco_sbi", prob_reco_sbi.__version__, "| torch", torch.__version__)

In [ ]:
# The figure style of the course (the paper style of the NeuroMCT figures): white, no grid, ticks pointing in on
# all four sides with minor ticks, framed legends inside the axes; samples as step histograms, a density compared with
# them dashed in the same colour.
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": ".15",
        "axes.labelcolor": ".15",
        "axes.linewidth": 1.25,
        "axes.grid": False,
        "axes.axisbelow": True,
        "axes.spines.left": True,
        "axes.spines.bottom": True,
        "axes.spines.right": True,
        "axes.spines.top": True,
        "text.color": ".15",
        "xtick.color": ".15",
        "ytick.color": ".15",
        "xtick.major.width": 1.25,
        "ytick.major.width": 1.25,
        "xtick.minor.width": 1.0,
        "ytick.minor.width": 1.0,
        "xtick.major.size": 6.0,
        "ytick.major.size": 6.0,
        "xtick.minor.size": 4.0,
        "ytick.minor.size": 4.0,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans", "Bitstream Vera Sans", "sans-serif"],
        "lines.linewidth": 1.5,
        "lines.markersize": 6.0,
        "patch.linewidth": 1.0,
        "patch.edgecolor": "w",
        "patch.force_edgecolor": True,
        "lines.solid_capstyle": "round",
        "legend.frameon": True,
        "image.cmap": "viridis",
        "xtick.minor.visible": True,
        "ytick.minor.visible": True,
        "xtick.direction": "in",
        "xtick.top": True,
        "xtick.bottom": True,
        "xtick.labelsize": 12,
        "ytick.direction": "in",
        "ytick.right": True,
        "ytick.left": True,
        "ytick.labelsize": 12,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "pdf.fonttype": 42,
    }
)
# One colour per meaning, the same in every figure of the course.
COLOURS = {
    "target": "royalblue",  # the camel p(x): its samples and its density
    "map": "black",  # maps: the learned one solid, the exact (oracle) one dashed
    "base": "teal",  # the base N(0, 1)
    "uniform": "peru",  # the uniform base U(0, 1)
    "planar": "darkgreen",  # the flow families (tasks F2-F4)
    "radial": "firebrick",
    "spline": "indigo",
}

## A flow on one axis

A flow is a strictly increasing map $z = f(x)$ of the real line together with a base distribution $p_Z$, here the
standard normal $\mathcal{N}(0, 1)$. It defines a density of $x$ by the change of variables

$$q(x) = p_Z\big(f(x)\big)\, f'(x), \qquad \log q(x) = \log p_Z\big(f(x)\big) + \log f'(x),$$

and it is sampled the other way round: draw $z \sim p_Z$ and return $x = f^{-1}(z)$. In the package every layer
returns $z$ together with $\log|\mathrm{d}z/\mathrm{d}x|$, and `Flow` sums these over its layers. A new spline layer
is the identity, so the flow below is exactly the base: its log-density is that of $\mathcal{N}(0, 1)$.

In [ ]:
flow = Flow([RQSpline(bins=16, bound=10.0)])  # one spline layer on [-10, 10], standard normal base
x_test = torch.linspace(-3.0, 3.0, 7)
z, logdet = flow(x_test)  # the map and log f'(x)
print("z = f(x):             ", z.detach().numpy().round(3) + 0.0)  # + 0.0 prints -0.0 as 0.0
print("log f'(x):            ", logdet.detach().numpy().round(3) + 0.0)
log_ratio = flow.log_prob(x_test) - StandardNormal().log_prob(x_test)
print("log q(x) - log phi(x):", log_ratio.detach().numpy().round(6) + 0.0)
print("inverse(f(x)) == x:   ", torch.allclose(flow.inverse(z), x_test))
print("parameters:", flow.n_params)

## The target: a camel with a shoulder

All examples learn the same mixture of three Gaussians: a narrow spike, a broad hump and a wide shoulder on the
right. Its density, CDF and quantile function are exact, so every fit can be compared with the truth. The parameters
are the dictionary `CAMEL`, and `Mixture1D(**CAMEL)` (or `camel()`) is the distribution.

In [ ]:
target = camel()
print(CAMEL)
print(f"mean {target.mean:.3f}, standard deviation {target.std:.3f}, entropy H(p) = {target.entropy():.4f} nats")

x = target.sample(20_000, seed=11)  # the training samples of the tasks: float64 tensor
grid = torch.linspace(-5.0, 9.0, 701, dtype=torch.float64)

fig, (left, right) = plt.subplots(1, 2, figsize=(9.0, 3.2), layout="constrained")
counts, edges = np.histogram(x.numpy(), bins=140, range=(-5, 9), density=True)
left.stairs(counts, edges, color=COLOURS["target"], alpha=0.6, label="20 000 samples")
left.plot(grid, target.pdf(grid), color=COLOURS["target"], ls="--", lw=1.25, label="target density $p(x)$")
left.set(xlabel="$x$", ylabel="Density", xlim=(-5, 9), ylim=(0, 0.6))
left.legend(loc="upper right")
right.plot(grid, target.cdf(grid), color=COLOURS["target"])
right.set(xlabel="$x$", ylabel="Cumulative probability $F(x)$", xlim=(-5, 9))
plt.show()

## The oracle map

For a continuous target with CDF $F$, the variable $u = F(x)$ is uniform on $(0, 1)$, and $z = \Phi^{-1}(F(x))$ is
standard normal. In one dimension this is the one increasing map that turns the target into the base: the exact flow,
which every learned flow can be compared with. `oracle_map(target)` returns it as a layer, so `Flow([oracle_map(target)])`
has $q = p$. The Kolmogorov-Smirnov test compares the mapped samples with $\mathcal{N}(0, 1)$; for an exact map its
p-value is uniformly distributed between 0 and 1.

In [ ]:
oracle = oracle_map(target)  # z = Phi^-1(F(x))
z_exact, _ = oracle(x)
print("Kolmogorov-Smirnov test of z against N(0, 1):", ks_statistic(z_exact))
print("KL to the target of the oracle flow:", kl_to_truth(Flow([oracle]), target), "nats")

t = np.linspace(-4, 4, 401)
fig, (left, middle, right) = plt.subplots(1, 3, figsize=(10.5, 3.4), layout="constrained")
left.plot(grid, oracle(grid)[0], color=COLOURS["map"])
left.set(xlabel="$x$", ylabel=r"$z = \Phi^{-1}(F(x))$", xlim=(-5, 9), title="The oracle map")
counts, edges = np.histogram(z_exact.numpy(), bins=60, range=(-4, 4), density=True)
middle.stairs(counts, edges, color=COLOURS["base"], alpha=0.6, label="mapped samples")
middle.plot(t, np.exp(-0.5 * t**2) / np.sqrt(2 * np.pi), color=COLOURS["base"], ls="--", lw=1.25, label="N(0, 1)")
middle.set(xlabel="$z$", ylabel="Density", xlim=(-4, 4), ylim=(0, 0.6), title="Mapped samples and N(0, 1)")
middle.legend(loc="upper right")
theoretical, empirical = qq_points(z_exact)
right.plot([-3, 3], [-3, 3], color="dimgray", ls="--", lw=0.8)
right.plot(theoretical, empirical, "o", ms=2.5, color=COLOURS["base"])
right.set(xlabel="Quantile of N(0, 1)", ylabel="Quantile of the mapped samples", title="Quantile-quantile plot")
plt.show()

## Fit a spline flow

`fit` trains a flow by maximum likelihood on samples: it minimises the mean negative log-likelihood
$-\frac{1}{N}\sum_i \log q(x_i)$ with Adam, which for many samples is $H(p) + \mathrm{KL}(p \,\|\, q)$. It needs
the samples only, never the density of the target: the situation of simulation-based inference. Every 10 steps it
records a snapshot (the loss, the parameters, and $q$ and $f$ on `grid`), the frames of the animation below.

In [ ]:
flow = Flow([RQSpline(bins=16, bound=10.0)])
history = fit(flow, x, steps=2000, lr=1e-2, every=10, grid=grid)
print(f"{flow.n_params} parameters, {len(history.step)} snapshots, {history.seconds:.1f} s")
print(f"training loss: {history.train_loss[0]:.4f} -> {history.train_loss[-1]:.4f} nats")

## How good is the fit: the KL divergence to the truth

In one dimension the KL divergence to the target is an ordinary integral, which `kl_to_truth` evaluates by quadrature
(in nats; zero only for $q = p$). From the samples alone, the same number is estimated by the training loss minus the
loss of the exact density on the same samples, $-\frac{1}{N}\sum_i \log p(x_i)$. The entropy $H(p)$ is not the right
reference: the mean over 20 000 samples fluctuates around its expectation by about
$\operatorname{std}(\log p)/\sqrt{N}$, which is more than the KL divergence of a good fit. On the training samples
the estimate is still biased low, because the flow was fitted to them. The learned map should follow the oracle map
wherever the target has samples.

In [ ]:
kl = kl_to_truth(flow, target)
with torch.no_grad():
    loss = -flow.log_prob(x).mean().item()  # the training loss now
reference = -target.log_pdf(x).mean().item()  # the loss of the exact density on the same samples
print(f"KL(p || q) = {kl:.4f} nats (quadrature);  loss - reference = {loss - reference:+.4f} nats (from the samples)")
print(f"loss {loss:.4f}, reference {reference:.4f}, entropy H(p) {target.entropy():.4f} nats")

with torch.no_grad():
    q = torch.exp(flow.log_prob(grid)).double()
    z_learned = flow(grid)[0].double()
fig, (left, right) = plt.subplots(1, 2, figsize=(9.0, 3.2), layout="constrained")
left.fill_between(grid, target.pdf(grid), color=COLOURS["target"], alpha=0.25, lw=0)
left.plot(grid, target.pdf(grid), color=COLOURS["target"], label="target $p(x)$")
left.plot(grid, q, color=COLOURS["spline"], label="flow $q(x)$")
left.set(xlabel="$x$", ylabel="Density", xlim=(-5, 9), ylim=(0, 0.6))
left.legend(loc="upper right")
right.plot(grid, z_learned, color=COLOURS["map"], label="learned map $f(x)$")
right.plot(grid, oracle(grid)[0], color=COLOURS["map"], ls="--", lw=1.25, label=r"oracle map $\Phi^{-1}(F(x))$")
right.set(xlabel="$x$", ylabel="$z$", xlim=(-5, 9), ylim=(-4.5, 4.5))
right.legend(loc="upper left")
plt.show()

Sampling runs the flow backwards, $x = f^{-1}(z)$ with $z \sim \mathcal{N}(0, 1)$, which is exact for the
spline. The Kolmogorov-Smirnov test also accepts the target as the reference.

In [ ]:
samples = flow.sample(20_000, seed=2)
print("Kolmogorov-Smirnov test of the flow's samples against the target:", ks_statistic(samples, target))

fig, ax = plt.subplots(figsize=(5.5, 3.4), layout="constrained")
counts, edges = np.histogram(samples.numpy(), bins=140, range=(-5, 9), density=True)
ax.stairs(counts, edges, color=COLOURS["spline"], alpha=0.6, label="samples of the flow")
ax.plot(grid, target.pdf(grid), color=COLOURS["target"], ls="--", lw=1.25, label="target $p(x)$")
ax.set(xlabel="$x$", ylabel="Density", xlim=(-5, 9), ylim=(0, 0.6))
ax.legend(loc="upper right")
plt.show()

## The training animation

Each frame is one snapshot of `history`: the model density next to the target, the learned map next to the oracle
map, and the loss. Here every fourth snapshot (every 40 steps) is drawn, to keep it quick; `FuncAnimation` redraws the
lines and `PillowWriter` saves the GIF.

In [ ]:
frames = range(0, len(history.step), 4)
fig, (left, middle, right) = plt.subplots(1, 3, figsize=(10.5, 3.4), layout="constrained")
left.fill_between(grid, target.pdf(grid), color=COLOURS["target"], alpha=0.25, lw=0)
left.plot(grid, target.pdf(grid), color=COLOURS["target"], label="target $p(x)$")
(line_q,) = left.plot([], [], color=COLOURS["spline"], label="flow $q(x)$")
left.set(xlabel="$x$", ylabel="Density", xlim=(-5, 9), ylim=(0, 0.6))
left.legend(loc="upper right")
(line_f,) = middle.plot([], [], color=COLOURS["map"], label="learned map")
middle.plot(grid, oracle(grid)[0], color=COLOURS["map"], ls="--", lw=1.25, label="oracle map")
middle.set(xlabel="$x$", ylabel="$z = f(x)$", xlim=(-5, 9), ylim=(-4.5, 4.5))
middle.legend(loc="upper left")
right.axhline(target.entropy(), color="black", ls="--", lw=1.0, label="entropy $H(p)$")
(line_loss,) = right.plot([], [], color=COLOURS["spline"], label="training loss")
right.set(xlabel="Adam step", ylabel="Training loss [nats]", xlim=(0, history.step[-1]), ylim=(1.8, 3.4))
right.legend(loc="upper right")


def draw(i):
    """Show snapshot i of the training."""
    line_q.set_data(history.grid, history.q_grid[i])
    line_f.set_data(history.grid, history.z_grid[i])
    line_loss.set_data(history.step[: i + 1], history.train_loss[: i + 1])
    left.set_title(f"step {history.step[i]}")
    return line_q, line_f, line_loss


animation = FuncAnimation(fig, draw, frames=frames)
animation.save(DATA / "flows_training.gif", writer=PillowWriter(fps=12), dpi=80)
plt.close(fig)
display(Image(filename=DATA / "flows_training.gif"))

## The particle animation: transport from the target to the base

`flow.transform_points(x)` returns the positions of samples after every layer: row 0 is $x$, the last row
$z = f(x)$. This flow has one layer, so every sample moves in one step; the frames interpolate between the two rows.
Coloured by their initial position (dark to light), the samples keep their order (the map is increasing): the narrow spike is spread
out and the broad shoulder is squeezed, until the histogram is $\mathcal{N}(0, 1)$. A flow with several layers
moves the samples layer by layer (task F6.2).

In [ ]:
points = flow.transform_points(target.sample(2000, seed=3)).double().numpy()  # shape [layers + 1, 2000]
height = np.random.default_rng(0).uniform(0, 1, points.shape[1])
shares = np.linspace(0, 1, 26)
t = np.linspace(-4, 4, 401)
position = LinearSegmentedColormap.from_list("position", ["midnightblue", "lightskyblue"])  # dark to light: initial x
edges = np.linspace(-5, 9, 113)
peak = max(np.histogram((1 - s) * points[0] + s * points[-1], bins=edges, density=True)[0].max() for s in shares)
fig, (top, bottom) = plt.subplots(2, 1, figsize=(8.0, 4.6), sharex=True, layout="constrained")
top.plot(grid, target.pdf(grid), color=COLOURS["target"], ls=":", label="target $p(x)$")
top.plot(t, np.exp(-0.5 * t**2) / np.sqrt(2 * np.pi), color=COLOURS["base"], ls="--", lw=1.25, label="base N(0, 1)")
histogram = top.stairs(np.zeros(len(edges) - 1), edges, color=COLOURS["spline"], alpha=0.6, label="samples")
top.set(ylabel="Density", ylim=(0, max(0.6, 1.05 * peak)))  # no frame is clipped
top.legend(loc="upper right")
dots = bottom.scatter(points[0], height, s=3, c=points[0], cmap=position, vmin=-3, vmax=6)
bottom.set(xlabel="Position of the samples", yticks=[], xlim=(-5, 9))
bottom.tick_params(axis="y", which="both", left=False, right=False)


def move(share):
    """Move the samples a fraction ``share`` of the way from x to z = f(x)."""
    now = (1 - share) * points[0] + share * points[-1]
    dots.set_offsets(np.column_stack([now, height]))
    histogram.set_data(np.histogram(now, bins=edges, density=True)[0], edges)
    top.set_title(f"{share:.0%} of the way from x to z = f(x)")
    return dots, histogram


animation = FuncAnimation(fig, move, frames=np.concatenate([shares, shares[::-1]]))
animation.save(DATA / "flows_particles.gif", writer=PillowWriter(fps=12), dpi=80)
plt.close(fig)
display(Image(filename=DATA / "flows_particles.gif"))

## The uniform base

The base can also be the uniform distribution on $(0, 1)$. A flow with this base ends with a `Sigmoid()` layer, the
logistic function, after the spline. Below it is only built and evaluated, untrained; task F5 trains it and compares
the two bases.

In [ ]:
uniform_flow = Flow([RQSpline(bins=16, bound=10.0), Sigmoid()], Uniform())  # untrained: the logistic function
with torch.no_grad():
    u = uniform_flow(x)[0]
print(f"u = f(x) of the samples lies in [{u.min():.2e}, {u.max():.6f}]")
print("samples of the untrained flow:", uniform_flow.sample(5, seed=4).numpy().round(3))

## Where this is used in a real experiment

A conditional one-dimensional flow of exactly this kind, $z = f(x, \psi)$ with a standard normal base and
trained by maximum likelihood on simulated events, is the NFDE likelihood model of the NeuroMCT paper
([Commun. Phys. 9, 63 (2026)](https://doi.org/10.1038/s42005-026-02499-6)): $x$ is the number of photo-electrons of a
calibration event in the JUNO detector and $\psi$ are the three energy-response parameters. The section
[Where this is used](../docs/flows.md#where-this-is-used-a-one-dimensional-flow-in-a-real-experiment) of
`docs/flows.md` shows the learned map, the flow building a calibration spectrum layer by layer, and the fitted
spectra of five sources. Bonus task B2 is the toy version.

## Next steps

- The tasks are in [TASKS_flows.md](../TASKS_flows.md): the target and the oracle map (F1), planar and radial flows
  written by hand (F2, F3), splines (F4), the uniform base (F5) and the animations (F6).
- [docs/flows.md](../docs/flows.md) has the maths of every transform and the reference figures.
- `python examples/flows_quickstart.py` runs the same fit from the command line.